# Used Car Price Prediction

## 1) Problem statement.

* This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
* If user can predict the price of the car based on input features.
* Prediction results can be used to give new seller the price suggestion based on market condition.

## 2) Data Collection.
* The Dataset is collected from scrapping from cardheko webiste
* The data consists of 13 column and 15411 rows.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [5]:
df=pd.read_csv("cardekho_imputated.csv")
df.drop(columns=["Unnamed: 0"],axis=1,inplace=True)
df

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


## Feature Engineering

In [6]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [7]:
df.columns

Index(['car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type',
       'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power',
       'seats', 'selling_price'],
      dtype='object')

In [8]:
df.drop(columns=['car_name','brand'],axis=1,inplace=True)

In [9]:
df

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...
15406,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [10]:
#Get all features
num_features=[feature for feature in df.columns if df[feature].dtype!='O']
print("Num of Numerical features: ",len(num_features))

cat_features=[feature for feature in df.columns if df[feature].dtype=='O']
print("Num of Categorical features: ",len(cat_features))

discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print("Num of Discrete features: ",len(discrete_features))

continous_features=[feature for feature in num_features if feature not in discrete_features]
print("Num of continous features: ",len(continous_features))


Num of Numerical features:  7
Num of Categorical features:  4
Num of Discrete features:  2
Num of continous features:  5


In [11]:
#Independent and dependent feature
X=df.drop(columns=['selling_price'],axis=1)
y=df['selling_price']

In [12]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
15406,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
15407,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
15408,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
15409,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [13]:
print(cat_features)

['model', 'seller_type', 'fuel_type', 'transmission_type']


In [14]:
len(df['model'].unique())

120

In [15]:
len(df['seller_type'].unique())

3

In [16]:
len(df['fuel_type'].unique())

5

In [17]:
len(df['transmission_type'].unique())

2

In [18]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X['model']=le.fit_transform(X['model'])

In [19]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
15406,117,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
15407,42,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
15408,77,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
15409,114,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [24]:
onehot_feature=[feature for feature in cat_features if feature!='model']
print(onehot_feature)
std_feature=X.select_dtypes(exclude='object').columns
print(std_feature)

['seller_type', 'fuel_type', 'transmission_type']
Index(['model', 'vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power',
       'seats'],
      dtype='object')


In [25]:

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

onehotencoder=OneHotEncoder(drop='first')
scaler=StandardScaler()

preprocessor=ColumnTransformer(
    [
        ('OneHotEncoder',onehotencoder,onehot_feature),
        ('StandardScaler',scaler,std_feature)
    ],
    remainder='passthrough'
)


In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [27]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)


In [29]:
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.598177,0.988218,-0.000504,-0.439588,-0.762989,-0.895572,-0.402521
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.241139,-0.010659,1.050645,-0.108487,0.932816,0.957290,-0.402521
2,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.296068,-0.343618,-0.152704,-0.852265,-0.545726,-0.616117,-0.402521
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.531966,-0.676576,-0.369307,-0.276437,-0.549571,-0.432239,-0.402521
4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.225756,-1.009535,-1.157621,-0.204458,-0.549571,-0.431535,-0.402521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11553,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.555329,0.322300,1.742114,0.244208,-0.451514,-0.268558,2.066875
11554,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.900285,1.654135,0.087966,-0.878657,0.221424,0.069607,-0.402521
11555,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.037607,0.322300,-0.850646,0.181826,-0.932185,-0.779563,-0.402521
11556,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.531966,-1.342494,-0.970981,-0.276437,-0.549571,-0.431535,-0.402521


In [30]:
X_train.shape, X_test.shape

((11558, 14), (3853, 14))

## Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error

In [32]:
#Function to evaluate model
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    rmse=root_mean_squared_error(true,predicted)
    score=r2_score(true,predicted)
    return mae,rmse,score

In [33]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
   
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    #Make predictions
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')


Linear Regression
Model performance for Training set
- Root Mean Squared Error: 552154.2495
- Mean Absolute Error: 266675.1076
- R2 Score: 0.6220
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 519891.2635
- Mean Absolute Error: 284283.4460
- R2 Score: 0.6525


Lasso
Model performance for Training set
- Root Mean Squared Error: 552154.2607
- Mean Absolute Error: 266674.0558
- R2 Score: 0.6220
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 519890.6459
- Mean Absolute Error: 284283.8047
- R2 Score: 0.6525


Ridge
Model performance for Training set
- Root Mean Squared Error: 552154.8771
- Mean Absolute Error: 266635.7643
- R2 Score: 0.6220
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 519881.2426
- Mean Absolute Error: 284241.5752
- R2 Score: 0.6525


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 325070.6406
- Mean 